# Notebook Title: Exploratory Data Analysis (EDA) - Student & Career Profiles

## Purpose
"Exploratory Data Analysis of raw and refined student profiles" 
To validate data integrity, identify underlying distributions across the 11-point vector space, and confirm the statistical readiness of the datasets for Machine Learning model training.

## Context
- **Project Phase:** Exploration / Research
- **Related Module(s):** `src/data_processing`, `src/research`

## Data Sources
- **External**
- **Raw Kaggle Dataset:** Student personality survey responses (TIPI and RIASEC).
- **Refined Vector Library:** Normalized 11-point vector profiles (6 RIASEC + 5 Big Five) derived from the raw Kaggle data and O*NET Career Work Styles.

## Assumptions & Constraints
- **Assumptions:** Survey responses are considered accurate reflections of user traits; O*NET mappings to Big Five are psychologically sound.
- **Constraints:** Analysis is limited to the 11 dimensions defined in the unified vector space; original Kaggle data may contain noise or missing values that require filtering.

## Reproducibility
- **Environment:** Local (VS Code / Jupyter / Anaconda)

## Expected Outputs
- **Distribution Plots:** Histograms showing the spread of RIASEC and Big Five scores.
- **Correlation Heatmap:** Visualizing the relationship between interests and personality traits.
- **PCA Visualization:** 2D mapping of the 11D vector space to identify clusters and data coverage.
- **Data Health Metrics:** Missing value counts and normalization range checks (0.0 to 1.0).

## Notes
- This notebook acts as a "Data Health Check" to ensure that the normalization and feature alignment between the Kaggle (Student) and O*NET (Career) datasets are mathematically consistent before training the Random Forest model.

In [1]:
import pandas as pd

# Update this path to where your Kaggle Raw file is saved
kaggle_raw_path = 'docs\data\Kaggle_Raw.csv'

try:
    kaggle_raw = pd.read_csv(kaggle_raw_path)
except FileNotFoundError:
    kaggle_raw = pd.read_csv(f"../{kaggle_raw_path}")

print("Kaggle Columns:")
print(kaggle_raw.columns.tolist())

# Look at the first 5 rows to see how they store the RIASEC data
display(kaggle_raw.head())

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
C:\Users\grosh\AppData\Local\Temp\ipykernel_63824\4203520460.py:4: SyntaxWarning: invalid escape sequence '\d'
  kaggle_raw_path = 'docs\data\Kaggle_Raw.csv'


Kaggle Columns:
['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'introelapse', 'testelapse', 'surveyelapse', 'TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 'VCL1', 'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL6', 'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL11', 'VCL12', 'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education', 'urban', 'gender', 'engnat', 'age', 'hand', 'religion', 'orientation', 'race', 'voted', 'married', 'familysize', 'uniqueNetworkLocation', 'country', 'source', 'major']


,R1,R2,R3,R4,R5,R6,R7,R8,I1,I2,...,religion,orientation,race,voted,married,familysize,uniqueNetworkLocation,country,source,major
0,3,4,3,1,1,4,1,3,5,5,...,7,1,1,2,1,1,1,US,2,NaN
1,1,1,2,4,1,2,2,1,5,5,...,7,3,4,1,2,3,1,US,1,Nursing
2,2,1,1,1,1,1,1,1,4,1,...,7,1,4,2,1,1,1,US,1,NaN
3,3,1,1,2,2,2,2,2,4,1,...,0,1,1,2,1,1,1,CN,0,NaN
4,4,1,1,2,1,1,1,2,5,5,...,4,3,1,2,1,4,1,PH,0,education


In [2]:
kaggle_raw.describe(include='all')

,R1,R2,R3,R4,R5,R6,R7,R8,I1,I2,...,religion,orientation,race,voted,married,familysize,uniqueNetworkLocation,country,source,major
count,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,...,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,1.458280e+05,145828.000000,145816,145828.000000,92954
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,188,NaN,15953
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,US,NaN,psychology
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80579,NaN,6861
mean,2.572846,2.106859,1.748738,2.293640,1.749945,2.197815,2.017020,1.967215,3.434752,3.335669,...,5.481375,1.363497,3.287846,1.620827,1.274913,1.255801e+05,1.309275,NaN,0.420242,NaN
std,1.318347,1.235213,1.120873,1.339156,1.072175,1.273549,1.182839,1.184509,1.333190,1.362869,...,3.417863,1.041039,1.400790,0.512136,0.566570,1.612271e+07,0.462196,NaN,0.651346,NaN
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,1.000000,NaN,0.000000,NaN
25%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.000000,2.000000,...,2.000000,1.000000,3.000000,1.000000,1.000000,2.000000e+00,1.000000,NaN,0.000000,NaN
50%,3.000000,2.000000,1.000000,2.000000,1.000000,2.000000,2.000000,2.000000,4.000000,4.000000,...,6.000000,1.000000,4.000000,2.000000,1.000000,3.000000e+00,1.000000,NaN,0.000000,NaN
75%,3.000000,3.000000,2.000000,3.000000,2.000000,3.000000,3.000000,3.000000,5.000000,4.000000,...,7.000000,1.000000,4.000000,2.000000,1.000000,3.000000e+00,2.000000,NaN,1.000000,NaN


In [3]:
kaggle_raw.isnull().sum()

R1                           0
R2                           0
R3                           0
R4                           0
R5                           0
                         ...  
familysize                   0
uniqueNetworkLocation        0
country                     12
source                       0
major                    52874
Length: 93, dtype: int64

In [4]:
kaggle_raw.columns

Index(['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'I1', 'I2', 'I3', 'I4',
       'I5', 'I6', 'I7', 'I8', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8',
       'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'E1', 'E2', 'E3', 'E4',
       'E5', 'E6', 'E7', 'E8', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8',
       'introelapse', 'testelapse', 'surveyelapse', 'TIPI1', 'TIPI2', 'TIPI3',
       'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 'VCL1',
       'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL6', 'VCL7', 'VCL8', 'VCL9', 'VCL10',
       'VCL11', 'VCL12', 'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education',
       'urban', 'gender', 'engnat', 'age', 'hand', 'religion', 'orientation',
       'race', 'voted', 'married', 'familysize', 'uniqueNetworkLocation',
       'country', 'source', 'major'],
      dtype='object')